# 用 CLIP embedding 搭配 GPT-4 Vision 強化多模態 RAG

> 原文：[examples/custom_image_embedding_search.ipynb](https://github.com/openai/openai-cookbook/blob/main/examples/custom_image_embedding_search.ipynb)



## 這篇在做什麼

這本 notebook 用 CLIP 把一批科技產品圖片直接轉成 embedding，建立圖片向量資料庫。使用者上傳一張圖，就能找出最相似的圖片，再把圖片和它的描述一起丟給 GPT-4 Vision 回答問題。重點是不先把圖片轉成文字說明，直接用圖片比對，檢索會比較準。

## 學完你會

- 用 CLIP 模型把一個資料夾的圖片轉成 embedding，建成簡單的向量資料庫。
- 對使用者上傳的圖片做相似度搜尋，取回最相近的兩張圖和對應描述。
- 把檢索到的圖片與描述當 context，交給 GPT-4 Vision 回答關於產品的問題。

## 開跑前

- 需要 OpenAI API key，用來呼叫 GPT-4 Vision。
- 需要自備圖片資料夾 `image_database`（JPEG）和對應的 `description.json`。
- 在 Colab 執行時，用 `!pip install` 裝套件，圖片和 json 要先上傳到 Colab 的檔案區。

In [ ]:
# 環境設定：在 Colab 左側 🔑 Secrets 新增 OPENAI_API_KEY
%pip install -q -U faiss-cpu git+https://github.com/openai/CLIP.git matplotlib numpy openai pillow torch tqdm
import os
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
except ImportError:
    pass  # 本機執行就用自己的環境變數

# 抓原 repo 的資料檔（只下載這個資料夾）
if not os.path.exists('openai-cookbook'):
    !git clone -q --depth 1 --filter=blob:none --sparse https://github.com/openai/openai-cookbook
    !cd openai-cookbook && git sparse-checkout set examples/data
os.chdir('openai-cookbook/examples')

# 用 CLIP embedding 和 GPT-4 Vision 做多模態 RAG


多模態 RAG 是把其他模態加進傳統純文字的 RAG，提供額外的 context 並讓文字資料有所依據，讓 LLM 回答問題時理解得更好。

這裡沿用[服飾搭配 cookbook](https://cookbook.openai.com/examples/how_to_combine_gpt4v_with_rag_outfit_assistant) 的做法，直接對圖片做 embedding 再比對相似度，跳過「先幫圖片寫文字說明」這個會損失資訊的步驟，提升檢索準確度。

用 CLIP 這類 embedding 還有一個好處，可以拿特定資料來 fine-tuning，也能更新成沒看過的新圖片。

這個技巧的示範情境是：使用者提供一張科技產品的圖片，去搜尋企業知識庫，回傳相關資訊。

# 安裝

先安裝需要的套件。

In [ ]:
#installations
%pip install clip
%pip install torch
%pip install pillow
%pip install faiss-cpu
%pip install numpy
%pip install git+https://github.com/openai/CLIP.git
%pip install openai

接著 import 所有會用到的套件。


In [3]:
# model imports
import faiss
import json
import torch
from openai import OpenAI
import torch.nn as nn
from torch.utils.data import DataLoader
import clip
client = OpenAI()

# helper imports
from tqdm import tqdm
import json
import os
import numpy as np
import pickle
from typing import List, Union, Tuple

# visualisation imports
from PIL import Image
import matplotlib.pyplot as plt
import base64

現在載入 CLIP 模型。

In [4]:
#load model on device. The device you are running inference/training on is either a CPU or GPU if you have.
device = "cpu"
model, preprocess = clip.load("ViT-B/32",device=device)


接下來會依序做：
1.   建立圖片 embedding 資料庫
2.   設定對 vision 模型的查詢
3.   執行語意搜尋
4.   把使用者的問題連同圖片一起送出



# 建立圖片 embedding 資料庫

接著從一個圖片資料夾建立圖片 embedding 知識庫。這就是我們要搜尋的科技產品知識庫，用來回答使用者上傳圖片時的提問。

我們傳入存放圖片（JPEG 格式）的資料夾，逐張迴圈建立 embedding。

另外還有一個 description.json，知識庫裡每一張圖都有一筆資料，包含兩個 key：'image_path' 和 'description'。它把每張圖對應到一段有用的描述，幫助回答使用者的問題。

先寫一個函式，取得指定資料夾裡所有圖片的路徑。然後從名為 'image_database' 的資料夾取出所有 jpeg。

In [5]:
def get_image_paths(directory: str, number: int = None) -> List[str]:
    image_paths = []
    count = 0
    for filename in os.listdir(directory):
        if filename.endswith('.jpeg'):
            image_paths.append(os.path.join(directory, filename))
            if number is not None and count == number:
                return [image_paths[-1]]
            count += 1
    return image_paths
direc = 'image_database/'
image_paths = get_image_paths(direc)


接著寫一個函式，給一串路徑，從 CLIP 模型取得圖片 embedding。

我們先用前面拿到的 preprocess 函式處理圖片，它會做縮放、正規化、調整色彩通道等，確保餵給 CLIP 的輸入格式和維度正確。

然後把處理好的圖片疊在一起，一次丟進模型，不用逐張跑迴圈。最後回傳模型輸出，也就是 embedding 陣列。

In [6]:
def get_features_from_image_path(image_paths):
  images = [preprocess(Image.open(image_path).convert("RGB")) for image_path in image_paths]
  image_input = torch.tensor(np.stack(images))
  with torch.no_grad():
    image_features = model.encode_image(image_input).float()
  return image_features
image_features = get_features_from_image_path(image_paths)


現在可以建立向量資料庫了。

In [7]:
index = faiss.IndexFlatIP(image_features.shape[1])
index.add(image_features)


再讀入 json，建立圖片與描述對應的 list。同時寫一個 helper 函式，在這個 list 裡找出指定圖片，取得它的描述。

In [11]:
data = []
image_path = 'train1.jpeg'
with open('description.json', 'r') as file:
    for line in file:
        data.append(json.loads(line))
def find_entry(data, key, value):
    for entry in data:
        if entry.get(key) == value:
            return entry
    return None

來顯示一張範例圖片，這會當成使用者上傳的圖。這是 2024 CES 上發表的產品，DELTA Pro Ultra 全屋電池發電機。

In [ ]:
im = Image.open(image_path)
plt.imshow(im)
plt.show()

![Delta Pro](https://raw.githubusercontent.com/openai/openai-cookbook/main/images/train1.jpeg)

# 查詢 vision 模型

先看看 GPT-4 Vision（它沒看過這項產品）會把這張圖標成什麼。



首先要寫一個函式，把圖片編碼成 base64，因為這是傳給 vision 模型的格式。然後寫一個通用的 image_query 函式，讓我們能用圖片輸入來查詢 LLM。

In [12]:
def encode_image(image_path):
    with open(image_path, 'rb') as image_file:
        encoded_image = base64.b64encode(image_file.read())
        return encoded_image.decode('utf-8')

def image_query(query, image_path):
    response = client.chat.completions.create(
        model='gpt-4-vision-preview',
        messages=[
            {
            "role": "user",
            "content": [
                {
                "type": "text",
                "text": query,
                },
                {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{encode_image(image_path)}",
                },
                }
            ],
            }
        ],
        max_tokens=300,
    )
    # Extract relevant features from the response
    return response.choices[0].message.content
image_query('Write a short label of what is show in this image?', image_path)

'Autonomous Delivery Robot'

可以看到，它會根據訓練時學到的東西盡力回答，但因為沒看過類似的東西所以答錯了。這張圖本身比較模糊，很難推論出正確答案。

# 執行語意搜尋

現在做相似度搜尋，從知識庫找出最相似的兩張圖。做法是對使用者提供的 `image_path` 做 embedding，取回資料庫中各圖片的索引和相似度分數。分數越大代表越相似，所以我們把分數由大到小排序。

In [13]:
image_search_embedding = get_features_from_image_path([image_path])
distances, indices = index.search(image_search_embedding.reshape(1, -1), 2) #2 signifies the number of topmost similar images to bring back
distances = distances[0]
indices = indices[0]
indices_distances = list(zip(indices, distances))
indices_distances.sort(key=lambda x: x[1], reverse=True)

我們用索引去 `image_directory` 找出對應的圖片，傳給 vision 模型做 RAG。

來看它找回了什麼（依相似度順序顯示）：

In [ ]:
#display similar images
for idx, distance in indices_distances:
    print(idx)
    path = get_image_paths(direc, idx)[0]
    im = Image.open(path)
    plt.imshow(im)
    plt.show()

![Delta Pro2](https://raw.githubusercontent.com/openai/openai-cookbook/main/images/train2.jpeg)

![Delta Pro3](https://raw.githubusercontent.com/openai/openai-cookbook/main/images/train17.jpeg)

可以看到它找回了兩張都含有 DELTA Pro Ultra 全屋電池發電機的圖片。其中一張的背景有些會干擾的東西，但還是成功找到正確的圖。

# 用最相似的圖片回答使用者問題

現在拿最相似的那張圖和它的描述，連同使用者的問題一起丟給 gpt-v，讓使用者詢問他可能買到的產品。這裡就是 vision 模型的威力所在：你可以問模型沒被明確訓練過的一般問題，它也能高準確度地回答。

下面的例子會問這個產品的容量。

In [14]:
similar_path = get_image_paths(direc, indices_distances[0][0])[0]
element = find_entry(data, 'image_path', similar_path)

user_query = 'What is the capacity of this item?'
prompt = f"""
Below is a user query, I want you to answer the query using the description and image provided.

user query:
{user_query}

description:
{element['description']}
"""
image_query(prompt, similar_path)

'The portable home battery DELTA Pro has a base capacity of 3.6kWh. This capacity can be expanded up to 25kWh with additional batteries. The image showcases the DELTA Pro, which has an impressive 3600W power capacity for AC output as well.'

結果它答得出來。這是因為我們直接比對圖片，再從中取得相關描述當 context 才做得到。

# 結論

這本 notebook 走過了如何使用 CLIP 模型：用它建立圖片 embedding 資料庫、執行語意搜尋，最後把使用者的問題連同結果送出去回答。

這種用法可以套用到很多不同領域，也很容易再改進。例如可以 fine-tune CLIP，可以像一般 RAG 一樣改善檢索流程，也可以對 GPT-V 做 prompt engineering。


## 重點整理

- 直接對圖片做 embedding 做檢索，比先轉成文字說明少損失資訊。
- 向量資料庫加上 description.json，讓檢索到的圖片能帶著文字 context 去回答問題。
- 相似度分數越大越相似，排序後取前兩名就是檢索結果。
- vision 模型沒看過的東西會答錯，補上檢索到的圖片與描述後才答得對，這就是 RAG 的價值。
- CLIP 可以 fine-tune，檢索流程也能照文字 RAG 的方式優化。

## 練習

1. 把相似度搜尋取前兩名的數字改成前三名，顯示結果看看第三張圖還相不相關。
2. 把最後對 GPT-4 Vision 的問題從「容量」改成別的問題，例如產品的輸出功率，觀察答案是否仍然來自描述。
3. 在 image_query 時只送使用者上傳的圖片，不附檢索到的圖片和描述，比較答案跟有 RAG 的版本差在哪。